
# SACAIR 2026 — BOOK 3: Reviewer-Hardening Validation

**Purpose:** strengthen the already-frozen Book 1 (Hilbot-FI) and Book 2 (BANKING77) experiments without changing either notebook.

This notebook addresses six reviewer-facing questions:

1. **Neural seed stability** — Hilbot V3 / Hybrid CNN+TF-IDF and DistilBERT at seeds `42, 123, 2026`.
2. **Multiplicity** — Holm correction for the 12 canonical paired accuracy tests and secondary uncertainty tests.
3. **Direct robustness comparison** — paired bootstrap confidence intervals for *difference in degradation* relative to calibrated SVM.
4. **Dataset audit** — train/test near-duplicate and normalized-template checks, plus source-stratified sanity checks.
5. **Calibration under perturbation** — ECE, multiclass Brier score, and NLL on matched clean vs perturbed queries.
6. **Perturbation validity audit** — export all abbreviation/shortening pairs and a fixed typo sample for manual semantic review.

### Frozen protocol
- **Do not regenerate the train/test split.**
- **Do not regenerate perturbations.**
- **Do not tune hyperparameters.**
- Validation indices are frozen with `random_state=42`.
- The only intentional change across neural runs is stochastic model training (`seed = 42, 123, 2026`).
- **Top-2 margin is the primary uncertainty score for cross-model comparison.**
- Maximum probability and negative entropy remain secondary diagnostics.

Book 1 and Book 2 remain the immutable evidence archive. If this notebook discovers a genuine error, flag it rather than silently changing the old notebooks.


In [ ]:

# ============================================================
# 1. ENVIRONMENT + IMPORTS
# ============================================================

# Book 1 was run with:
# TensorFlow 2.20.0
# PyTorch 2.11.0+cpu
# Transformers 5.13.1
#
# We force CPU here to make the seed-42 rerun as comparable as possible
# to the recorded DistilBERT run in Book 1.

import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import gc
import re
import json
import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import tensorflow as tf
import torch
import transformers

from scipy.stats import binomtest, wilcoxon

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    log_loss
)
from sklearn.metrics.pairwise import cosine_similarity

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Input, Embedding, Conv1D, GlobalMaxPooling1D,
    Dense, Dropout, Concatenate
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

from torch.utils.data import TensorDataset, DataLoader
from torch.nn import CrossEntropyLoss
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    set_seed as hf_set_seed
)

warnings.filterwarnings("ignore", category=FutureWarning)

print("TensorFlow :", tf.__version__)
print("PyTorch    :", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA visible to PyTorch:", torch.cuda.is_available())

EXPECTED = {
    "tensorflow": "2.20.0",
    "transformers": "5.13.1",
}
print("\nReference Book-1 versions:", EXPECTED)


TensorFlow : 2.20.0
PyTorch    : 2.11.0+cu128
Transformers: 5.13.1
CUDA visible to PyTorch: False

Reference Book-1 versions: {'tensorflow': '2.20.0', 'transformers': '5.13.1'}


In [ ]:

# ============================================================
# 2. MOUNT DRIVE + LOAD FROZEN DATA
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/Hilbot chatbot (1)/Hilbot-FI")
GLOVE_PATH = Path("/content/drive/MyDrive/Hilbot/glove.6B.200d.txt")

OUT_DIR = BASE_DIR / "book3_reviewer_hardening"
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(BASE_DIR / "train.csv")
test_df = pd.read_csv(BASE_DIR / "test.csv")
full_df = pd.read_csv(BASE_DIR / "full_dataset.csv")

# Frozen perturbations produced by Book 1
typo_eval = pd.read_csv(BASE_DIR / "test_typo.csv")
abbrev_eval = pd.read_csv(BASE_DIR / "test_abbreviation.csv")
short_eval = pd.read_csv(BASE_DIR / "test_shortened.csv")

clean_typo = pd.read_csv(BASE_DIR / "clean_matched_typo.csv")
clean_abbrev = pd.read_csv(BASE_DIR / "clean_matched_abbreviation.csv")
clean_short = pd.read_csv(BASE_DIR / "clean_matched_shortened.csv")

print("Train:", train_df.shape)
print("Test :", test_df.shape)
print("Full :", full_df.shape)
print("Typo changed:", typo_eval.shape)
print("Abbreviation changed:", abbrev_eval.shape)
print("Shortening changed:", short_eval.shape)
print("GloVe exists:", GLOVE_PATH.exists())
print("Output directory:", OUT_DIR)


Mounted at /content/drive
Train: (1220, 4)
Test : (305, 4)
Full : (1525, 4)
Typo changed: (303, 5)
Abbreviation changed: (106, 5)
Shortening changed: (116, 5)
GloVe exists: True
Output directory: /content/drive/MyDrive/Hilbot chatbot (1)/Hilbot-FI/book3_reviewer_hardening


In [ ]:

# ============================================================
# 3. HARD INVARIANTS — STOP IF THESE FAIL
# ============================================================

assert len(train_df) == 1220
assert len(test_df) == 305
assert len(full_df) == 1525
assert train_df["label"].nunique() == 33
assert test_df["label"].nunique() == 33

assert len(typo_eval) == 303
assert len(abbrev_eval) == 106
assert len(short_eval) == 116

assert len(clean_typo) == len(typo_eval)
assert len(clean_abbrev) == len(abbrev_eval)
assert len(clean_short) == len(short_eval)

assert np.array_equal(
    clean_typo["label"].astype(str).values,
    typo_eval["label"].astype(str).values
)
assert np.array_equal(
    clean_abbrev["label"].astype(str).values,
    abbrev_eval["label"].astype(str).values
)
assert np.array_equal(
    clean_short["label"].astype(str).values,
    short_eval["label"].astype(str).values
)

# Clean matched text must be exactly the original text stored in the perturbed files.
assert np.array_equal(
    clean_typo["text"].astype(str).values,
    typo_eval["original_text"].astype(str).values
)
assert np.array_equal(
    clean_abbrev["text"].astype(str).values,
    abbrev_eval["original_text"].astype(str).values
)
assert np.array_equal(
    clean_short["text"].astype(str).values,
    short_eval["original_text"].astype(str).values
)

train_texts = set(train_df["text"].astype(str))
test_texts = set(test_df["text"].astype(str))
assert len(train_texts.intersection(test_texts)) == 0

print("✓ All frozen-data invariants passed.")


✓ All frozen-data invariants passed.


In [ ]:

# ============================================================
# 4. FIXED REPRESENTATIONS + FIXED VALIDATION INDICES
# ============================================================

FIXED_SPLIT_SEED = 42
NEURAL_SEEDS = [42, 123, 2026]

X_train_text = train_df["text"].astype(str)
X_test_text = test_df["text"].astype(str)
y_train = train_df["label"].astype(str)
y_test = test_df["label"].astype(str)

# Same TF-IDF protocol as Book 1
tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1
)
X_train_vec = tfidf.fit_transform(X_train_text)
X_test_vec = tfidf.transform(X_test_text)

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
NUM_CLASSES = len(label_encoder.classes_)

# IMPORTANT: validation partition stays frozen at seed 42 for ALL neural runs.
all_train_indices = np.arange(len(train_df))
train_idx, val_idx = train_test_split(
    all_train_indices,
    test_size=0.15,
    random_state=FIXED_SPLIT_SEED,
    stratify=y_train_encoded
)

print("TF-IDF dimension:", X_train_vec.shape[1])
print("Classes:", NUM_CLASSES)
print("Fixed neural train:", len(train_idx))
print("Fixed neural val  :", len(val_idx))

# All evaluation conditions, with matched clean controls.
CONDITIONS = {
    "clean_full": test_df.reset_index(drop=True),
    "clean_typo": clean_typo.reset_index(drop=True),
    "typo": typo_eval.reset_index(drop=True),
    "clean_abbrev": clean_abbrev.reset_index(drop=True),
    "abbreviation": abbrev_eval.reset_index(drop=True),
    "clean_short": clean_short.reset_index(drop=True),
    "shortening": short_eval.reset_index(drop=True),
}

PAIRS = {
    "typo": ("clean_typo", "typo"),
    "abbreviation": ("clean_abbrev", "abbreviation"),
    "shortening": ("clean_short", "shortening"),
}


TF-IDF dimension: 2589
Classes: 33
Fixed neural train: 1037
Fixed neural val  : 183


In [ ]:

# ============================================================
# 5. METRIC HELPERS
# ============================================================

def risk_coverage_curve(correct, certainty):
    correct = np.asarray(correct).astype(int)
    certainty = np.asarray(certainty)

    order = np.argsort(-certainty)
    correct_sorted = correct[order]
    n = len(correct_sorted)

    coverage = np.arange(1, n + 1) / n
    cumulative_correct = np.cumsum(correct_sorted)
    selective_accuracy = cumulative_correct / np.arange(1, n + 1)
    risk = 1.0 - selective_accuracy

    return pd.DataFrame({
        "coverage": coverage,
        "risk": risk,
        "selective_accuracy": selective_accuracy
    })


def aurc_from_score(correct, certainty):
    rc = risk_coverage_curve(correct, certainty)
    return float(np.trapezoid(rc["risk"].values, rc["coverage"].values))


def multiclass_brier_score(y_true, probs, classes):
    class_to_idx = {c: i for i, c in enumerate(classes)}
    y_onehot = np.zeros_like(probs, dtype=float)

    for i, label in enumerate(y_true):
        y_onehot[i, class_to_idx[label]] = 1.0

    return float(np.mean(np.sum((probs - y_onehot) ** 2, axis=1)))


def expected_calibration_error(y_true, y_pred, confidence, n_bins=10):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    confidence = np.asarray(confidence)
    correct = (y_true == y_pred).astype(float)

    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == n_bins - 1:
            mask = (confidence >= lo) & (confidence <= hi)
        else:
            mask = (confidence >= lo) & (confidence < hi)

        if mask.sum() == 0:
            continue

        ece += (mask.sum() / len(confidence)) * abs(
            correct[mask].mean() - confidence[mask].mean()
        )

    return float(ece)


def evaluate_probabilities(model_name, seed, condition_name, eval_df, probs, classes):
    y_true = eval_df["label"].astype(str).to_numpy()
    pred_idx = np.argmax(probs, axis=1)
    y_pred = np.asarray(classes)[pred_idx]

    sorted_probs = np.sort(probs, axis=1)
    confidence = sorted_probs[:, -1]
    margin = sorted_probs[:, -1] - sorted_probs[:, -2]
    entropy = -np.sum(probs * np.log(probs + 1e-12), axis=1)
    correct = y_pred == y_true

    if len(np.unique(correct)) == 2:
        auc_conf = roc_auc_score(correct.astype(int), confidence)
        auc_margin = roc_auc_score(correct.astype(int), margin)
        auc_entropy = roc_auc_score(correct.astype(int), -entropy)
    else:
        auc_conf = auc_margin = auc_entropy = np.nan

    metrics = {
        "model": model_name,
        "seed": seed,
        "condition": condition_name,
        "n": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "ece": expected_calibration_error(y_true, y_pred, confidence, n_bins=10),
        "brier": multiclass_brier_score(y_true, probs, classes),
        "nll": log_loss(y_true, probs, labels=classes),
        "mean_confidence": confidence.mean(),
        "mean_margin": margin.mean(),
        "mean_entropy": entropy.mean(),
        "auroc_confidence": auc_conf,
        "auroc_margin": auc_margin,
        "auroc_entropy": auc_entropy,
        "aurc_confidence": aurc_from_score(correct, confidence),
        "aurc_margin": aurc_from_score(correct, margin),
        "aurc_entropy": aurc_from_score(correct, -entropy),
    }

    q = pd.DataFrame({
        "model": model_name,
        "seed": seed,
        "condition": condition_name,
        "row_id": np.arange(len(eval_df)),
        "source": eval_df["source"].astype(str).values
                  if "source" in eval_df.columns else "unknown",
        "text": eval_df["text"].astype(str).values,
        "true_label": y_true,
        "predicted_label": y_pred,
        "correct": correct,
        "confidence": confidence,
        "margin": margin,
        "entropy": entropy,
    })

    return q, metrics


def holm_adjust(p_values):
    """Holm step-down adjusted p-values."""
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    order = np.argsort(p)
    p_sorted = p[order]

    adjusted_sorted = np.empty(m, dtype=float)
    running_max = 0.0

    for j, value in enumerate(p_sorted):
        candidate = (m - j) * value
        running_max = max(running_max, candidate)
        adjusted_sorted[j] = min(running_max, 1.0)

    adjusted = np.empty(m, dtype=float)
    adjusted[order] = adjusted_sorted
    return adjusted


def safe_wilcoxon_delta(clean_values, pert_values):
    clean_values = np.asarray(clean_values)
    pert_values = np.asarray(pert_values)
    delta = pert_values - clean_values

    if np.allclose(delta, 0):
        return 1.0, float(delta.mean())

    try:
        p = wilcoxon(delta, alternative="two-sided").pvalue
    except ValueError:
        p = np.nan

    return float(p), float(delta.mean())


print("✓ Metric helpers loaded.")


✓ Metric helpers loaded.



## Stage A — Dataset integrity audits

Run these **before neural training**. They are fast and can reveal whether a reviewer could reasonably argue that the structured portion of Hilbot-FI contains near-template leakage.


In [ ]:

# ============================================================
# 6. NEAR-DUPLICATE + TEMPLATE AUDIT
# ============================================================

def normalize_template(text):
    s = str(text).lower()
    # Replace numeric values while retaining lexical structure.
    s = re.sub(r"\d+(?:[.,]\d+)?", " <NUM> ", s)
    s = re.sub(r"[^a-z<>]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

train_raw = train_df["text"].astype(str).reset_index(drop=True)
test_raw = test_df["text"].astype(str).reset_index(drop=True)

# Diagnostic-only vectorizers may see train + test because they are NOT model features.
all_raw = pd.concat([train_raw, test_raw], ignore_index=True)

word_audit_vec = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1
)
char_audit_vec = TfidfVectorizer(
    analyzer="char_wb",
    lowercase=True,
    ngram_range=(3, 5),
    min_df=1
)

word_all = word_audit_vec.fit_transform(all_raw)
char_all = char_audit_vec.fit_transform(all_raw)

n_train = len(train_df)
word_train, word_test = word_all[:n_train], word_all[n_train:]
char_train, char_test = char_all[:n_train], char_all[n_train:]

word_sim = cosine_similarity(word_test, word_train)
char_sim = cosine_similarity(char_test, char_train)

word_best_idx = np.argmax(word_sim, axis=1)
char_best_idx = np.argmax(char_sim, axis=1)

duplicate_audit = pd.DataFrame({
    "test_index": np.arange(len(test_df)),
    "test_text": test_df["text"].astype(str).values,
    "test_label": test_df["label"].astype(str).values,
    "test_source": test_df["source"].astype(str).values,
    "word_max_similarity": word_sim[np.arange(len(test_df)), word_best_idx],
    "word_best_train_text": train_df.iloc[word_best_idx]["text"].astype(str).values,
    "word_best_train_label": train_df.iloc[word_best_idx]["label"].astype(str).values,
    "word_best_train_source": train_df.iloc[word_best_idx]["source"].astype(str).values,
    "char_max_similarity": char_sim[np.arange(len(test_df)), char_best_idx],
    "char_best_train_text": train_df.iloc[char_best_idx]["text"].astype(str).values,
    "char_best_train_label": train_df.iloc[char_best_idx]["label"].astype(str).values,
    "char_best_train_source": train_df.iloc[char_best_idx]["source"].astype(str).values,
})

duplicate_audit["word_best_same_label"] = (
    duplicate_audit["test_label"] == duplicate_audit["word_best_train_label"]
)
duplicate_audit["char_best_same_label"] = (
    duplicate_audit["test_label"] == duplicate_audit["char_best_train_label"]
)

train_templates = train_df["text"].map(normalize_template)
test_templates = test_df["text"].map(normalize_template)
train_template_set = set(train_templates)

duplicate_audit["normalized_template"] = test_templates.values
duplicate_audit["template_exact_train_overlap"] = test_templates.isin(train_template_set).values

def sim_summary(series):
    return {
        "median": series.median(),
        "p90": series.quantile(0.90),
        "p95": series.quantile(0.95),
        "max": series.max(),
        "fraction_gt_0.80": (series > 0.80).mean(),
        "fraction_gt_0.90": (series > 0.90).mean(),
        "fraction_gt_0.95": (series > 0.95).mean(),
    }

audit_rows = []

for source_name, subset in [
    ("ALL", duplicate_audit),
    ("csv", duplicate_audit[duplicate_audit["test_source"].str.lower() == "csv"]),
    ("json", duplicate_audit[duplicate_audit["test_source"].str.lower() == "json"]),
]:
    for metric in ["word_max_similarity", "char_max_similarity"]:
        row = {
            "source": source_name,
            "metric": metric,
            "n": len(subset),
            **sim_summary(subset[metric])
        }
        audit_rows.append(row)

duplicate_summary = pd.DataFrame(audit_rows)

print("SIMILARITY SUMMARY")
display(duplicate_summary.round(4))

template_summary = (
    duplicate_audit
    .groupby("test_source", dropna=False)
    .agg(
        n=("test_index", "size"),
        exact_template_overlap=("template_exact_train_overlap", "sum")
    )
    .reset_index()
)
template_summary["overlap_rate"] = (
    template_summary["exact_template_overlap"] / template_summary["n"]
)

print("\nNORMALIZED TEMPLATE OVERLAP")
display(template_summary.round(4))

print("\nTOP 25 WORD-TFIDF NEAR NEIGHBORS")
display(
    duplicate_audit
    .sort_values("word_max_similarity", ascending=False)
    .head(25)[[
        "test_source", "test_label", "test_text",
        "word_max_similarity", "word_best_train_label",
        "word_best_train_source", "word_best_train_text"
    ]]
)

duplicate_audit.to_csv(OUT_DIR / "near_duplicate_query_audit.csv", index=False)
duplicate_summary.to_csv(OUT_DIR / "near_duplicate_summary.csv", index=False)
template_summary.to_csv(OUT_DIR / "template_overlap_summary.csv", index=False)

print("\nSaved near-duplicate audit files.")


SIMILARITY SUMMARY


,source,metric,n,median,p90,p95,max,fraction_gt_0.80,fraction_gt_0.90,fraction_gt_0.95
0,ALL,word_max_similarity,305,0.5258,0.7884,0.8309,0.8953,0.0852,0.0000,0.0000
1,ALL,char_max_similarity,305,0.8104,0.9161,0.9263,0.9717,0.5508,0.1803,0.0262
2,csv,word_max_similarity,180,0.5435,0.7648,0.7706,0.7935,0.0000,0.0000,0.0000
3,csv,char_max_similarity,180,0.8480,0.9110,0.9182,0.9274,0.7167,0.1500,0.0000
4,json,word_max_similarity,125,0.4376,0.8412,0.8669,0.8953,0.2080,0.0000,0.0000
5,json,char_max_similarity,125,0.5695,0.9293,0.9541,0.9717,0.3120,0.2240,0.0640



NORMALIZED TEMPLATE OVERLAP


,test_source,n,exact_template_overlap,overlap_rate
0,csv,180,180,1.0
1,json,125,0,0.0



TOP 25 WORD-TFIDF NEAR NEIGHBORS


,test_source,test_label,test_text,word_max_similarity,word_best_train_label,word_best_train_source,word_best_train_text
61,json,monthly income (category),How much money did I earn in total from other ...,0.895256,yearly income (category),json,How much money did I earn in total from other ...
64,json,monthly savings (category),How much was save in mutual funds in total in ...,0.889880,yearly savings (category),json,How much was save in mutual funds in total in ...
57,json,yearly savings (category),What was the total amount save in emergency fu...,0.875645,monthly savings (category),json,What was the total amount save in emergency fu...
60,json,monthly income (category),I want to know my total income from other sour...,0.869016,yearly income (category),json,I want to know my total income from other sour...
63,json,monthly savings (category),I'd like to know my total savings in emergency...,0.867546,yearly savings (category),json,I'd like to know my total savings in emergency...
56,json,yearly savings (category),Can you tell me my total savings in fixed depo...,0.867259,monthly savings (category),json,Can you tell me my total savings in fixed depo...
50,json,yearly expense (category),I'd like to know my total expense on tithes fo...,0.867009,monthly expense (category),json,I'd like to know my total expense on tithes fo...
65,json,monthly expense (category),Can you tell me my total expense on health for...,0.866548,yearly expense (category),json,Can you tell me my total expense on health for...
52,json,yearly income (category),Can you tell me my total income from other sou...,0.865971,monthly income (category),json,Can you tell me my total income from other sou...
59,json,monthly income (category),What's the total amount earned from other sour...,0.860785,yearly income (category),json,What's the total amount earned from other sour...



Saved near-duplicate audit files.


In [ ]:

# ============================================================
# 7. PERTURBATION SEMANTIC-AUDIT EXPORT
# ============================================================

# Review ALL abbreviation and shortening pairs.
audit_abbrev = pd.DataFrame({
    "condition": "abbreviation",
    "original_text": clean_abbrev["text"].astype(str),
    "perturbed_text": abbrev_eval["text"].astype(str),
    "label": abbrev_eval["label"].astype(str),
    "source": abbrev_eval["source"].astype(str),
})

audit_short = pd.DataFrame({
    "condition": "shortening",
    "original_text": clean_short["text"].astype(str),
    "perturbed_text": short_eval["text"].astype(str),
    "label": short_eval["label"].astype(str),
    "source": short_eval["source"].astype(str),
})

# Fixed 100-query typo sample for human inspection.
rng = np.random.default_rng(42)
typo_sample_idx = np.sort(
    rng.choice(len(typo_eval), size=min(100, len(typo_eval)), replace=False)
)

audit_typo = pd.DataFrame({
    "condition": "typo",
    "original_text": clean_typo.iloc[typo_sample_idx]["text"].astype(str).values,
    "perturbed_text": typo_eval.iloc[typo_sample_idx]["text"].astype(str).values,
    "label": typo_eval.iloc[typo_sample_idx]["label"].astype(str).values,
    "source": typo_eval.iloc[typo_sample_idx]["source"].astype(str).values,
})

semantic_audit = pd.concat(
    [audit_abbrev, audit_short, audit_typo],
    ignore_index=True
)

# Fill these manually:
# 1 = intent clearly preserved
# 0 = intent not preserved
# 2 = uncertain / arguable
semantic_audit["valid_semantics"] = ""
semantic_audit["review_notes"] = ""

SEMANTIC_AUDIT_PATH = OUT_DIR / "perturbation_semantic_audit_TO_REVIEW.csv"
semantic_audit.to_csv(SEMANTIC_AUDIT_PATH, index=False)

print("Rows requiring semantic review:", len(semantic_audit))
print(semantic_audit["condition"].value_counts())
print("\nSaved to:")
print(SEMANTIC_AUDIT_PATH)

display(semantic_audit.head(15))


Rows requiring semantic review: 322
condition
shortening      116
abbreviation    106
typo            100
Name: count, dtype: int64

Saved to:
/content/drive/MyDrive/Hilbot chatbot (1)/Hilbot-FI/book3_reviewer_hardening/perturbation_semantic_audit_TO_REVIEW.csv


,condition,original_text,perturbed_text,label,source,valid_semantics,review_notes
0,abbreviation,How's your day?,hw's ur day?,concern,json,,
1,abbreviation,How's your day shaping up?,hw's ur day shaping up?,concern,json,,
2,abbreviation,Things are going smooth for now.,Things r going smooth for now.,current mood,json,,
3,abbreviation,Can't complain about anything today.,Can't complain abt anything tdy.,current mood,json,,
4,abbreviation,Thanks,thx,thanks,json,,
5,abbreviation,What can you do to help?,wht can u do to help?,options,json,,
6,abbreviation,What's your expertise?,wht's ur expertise?,options,json,,
7,abbreviation,How much did I earn from main source for Janua...,hw mch did I earn from main source for January...,income overview,json,,
8,abbreviation,What was my revenue from other sources in June...,wht was my revenue from other sources in June ...,income overview,json,,
9,abbreviation,Can you share my income in main source for the...,Can u share my inc in main source for the mnth...,income overview,json,,



## Stage B — Classical reference pipelines

These are rerun **only inside Book 3** to create query-level reference predictions needed for Holm correction, direct degradation comparisons, calibration-under-shift, and source audits. Their settings are unchanged from Book 1.


In [ ]:

# ============================================================
# 8. FIT FROZEN CLASSICAL PIPELINES
# ============================================================

lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)
lr.fit(X_train_vec, y_train)

base_svm = LinearSVC(
    class_weight="balanced",
    max_iter=5000,
    random_state=42
)
svm = CalibratedClassifierCV(
    base_svm,
    method="sigmoid",
    cv=3
)
svm.fit(X_train_vec, y_train)

print("✓ LR and calibrated SVM fitted.")


✓ LR and calibrated SVM fitted.


In [ ]:

# ============================================================
# 9. EVALUATE CLASSICAL PIPELINES ON ALL FROZEN CONDITIONS
# ============================================================

def evaluate_sklearn_all(model_name, model):
    query_frames = []
    metric_rows = []

    for condition_name, eval_df in CONDITIONS.items():
        X = tfidf.transform(eval_df["text"].astype(str))
        probs = model.predict_proba(X)

        q, m = evaluate_probabilities(
            model_name=model_name,
            seed=42,
            condition_name=condition_name,
            eval_df=eval_df,
            probs=probs,
            classes=model.classes_,
        )

        query_frames.append(q)
        metric_rows.append(m)

    return pd.concat(query_frames, ignore_index=True), pd.DataFrame(metric_rows)


lr_queries, lr_metrics = evaluate_sklearn_all("LR", lr)
svm_queries, svm_metrics = evaluate_sklearn_all("Calibrated SVM", svm)

classical_queries = pd.concat([lr_queries, svm_queries], ignore_index=True)
classical_metrics = pd.concat([lr_metrics, svm_metrics], ignore_index=True)

classical_queries.to_csv(OUT_DIR / "classical_query_results.csv", index=False)
classical_metrics.to_csv(OUT_DIR / "classical_condition_metrics.csv", index=False)

display(
    classical_metrics[
        classical_metrics["condition"] == "clean_full"
    ][[
        "model", "accuracy", "macro_f1", "weighted_f1",
        "ece", "brier", "nll", "auroc_margin", "aurc_margin"
    ]].round(4)
)


,model,accuracy,macro_f1,weighted_f1,ece,brier,nll,auroc_margin,aurc_margin
0,LR,0.7770,0.5390,0.7943,0.5547,0.6814,1.7771,0.7950,0.0935
7,Calibrated SVM,0.8164,0.5651,0.8166,0.1263,0.2552,0.6750,0.9243,0.0319



## Stage C — Multi-seed neural validation

**Important:** Seeds vary only model stochasticity. The train/validation/test partitions and perturbation sets remain fixed.

The notebook checkpoints each model/seed to CSV. If Colab disconnects, rerun from the top; already-completed seed runs will be loaded from disk unless `FORCE_RERUN=True`.


In [ ]:

# ============================================================
# 10. STATIC HYBRID INPUTS + GLOVE
# ============================================================

MAX_LEN = 30
EMBED_DIM = 200

hybrid_tokenizer = Tokenizer(oov_token="<OOV>")
hybrid_tokenizer.fit_on_texts(train_df["text"].astype(str))

train_sequences = hybrid_tokenizer.texts_to_sequences(train_df["text"].astype(str))
test_sequences = hybrid_tokenizer.texts_to_sequences(test_df["text"].astype(str))

X_seq_train = pad_sequences(
    train_sequences,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)
X_seq_test = pad_sequences(
    test_sequences,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_tfidf_train = X_train_vec.toarray().astype("float32")
X_tfidf_test = X_test_vec.toarray().astype("float32")
TFIDF_DIM = X_tfidf_train.shape[1]
VOCAB_SIZE = len(hybrid_tokenizer.word_index) + 1

seq_train = X_seq_train[train_idx]
seq_val = X_seq_train[val_idx]
tfidf_train = X_tfidf_train[train_idx]
tfidf_val = X_tfidf_train[val_idx]
y_h_train = y_train_encoded[train_idx]
y_h_val = y_train_encoded[val_idx]

classes_present = np.unique(y_h_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes_present,
    y=y_h_train
)
hilbot_class_weights = {
    int(c): float(w)
    for c, w in zip(classes_present, weights)
}

# Load GloVe only once.
embeddings_index = {}
with open(GLOVE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        values = line.rstrip().split(" ")
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        if len(vector) == EMBED_DIM:
            embeddings_index[word] = vector

glove_found = sum(
    1 for word in hybrid_tokenizer.word_index
    if word in embeddings_index
)

print("Hybrid vocabulary:", VOCAB_SIZE)
print("GloVe found:", glove_found)
print("GloVe coverage:", round(100 * glove_found / (VOCAB_SIZE - 1), 2), "%")
print("Fixed train / val:", len(train_idx), "/", len(val_idx))


Hybrid vocabulary: 595
GloVe found: 569
GloVe coverage: 95.79 %
Fixed train / val: 1037 / 183


In [ ]:

# ============================================================
# 11. HYBRID BUILD / TRAIN / EVALUATE
# ============================================================

def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    torch.manual_seed(seed)
    hf_set_seed(seed)


def make_embedding_matrix(seed):
    rng = np.random.default_rng(seed)
    matrix = rng.normal(
        loc=0.0,
        scale=0.6,
        size=(VOCAB_SIZE, EMBED_DIM)
    ).astype("float32")
    matrix[0] = 0.0

    for word, index in hybrid_tokenizer.word_index.items():
        vector = embeddings_index.get(word)
        if vector is not None:
            matrix[index] = vector

    return matrix


def build_hybrid(seed):
    tf.keras.backend.clear_session()
    set_all_seeds(seed)
    embedding_matrix = make_embedding_matrix(seed)

    seq_input = Input(shape=(MAX_LEN,), name="sequence_input")
    x = Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBED_DIM,
        weights=[embedding_matrix],
        trainable=False,
        name="frozen_glove"
    )(seq_input)
    x = Conv1D(
        filters=64,
        kernel_size=3,
        activation="relu",
        padding="same",
        name="glove_conv"
    )(x)
    x = GlobalMaxPooling1D(name="glove_global_pool")(x)

    tfidf_input = Input(shape=(TFIDF_DIM,), name="tfidf_input")
    t = Dense(256, activation="relu", name="tfidf_dense_256")(tfidf_input)
    t = Dense(128, activation="relu", name="tfidf_dense_128")(t)

    fusion = Concatenate(name="late_fusion")([x, t])
    fusion = Dense(128, activation="relu", name="fusion_dense")(fusion)
    fusion = Dropout(0.25, name="fusion_dropout")(fusion)
    output = Dense(NUM_CLASSES, activation="softmax", name="intent_output")(fusion)

    model = Model(
        inputs=[seq_input, tfidf_input],
        outputs=output,
        name=f"Hybrid_seed_{seed}"
    )
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


def hybrid_probs_for_df(model, eval_df):
    texts = eval_df["text"].astype(str)

    seq = pad_sequences(
        hybrid_tokenizer.texts_to_sequences(texts),
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

    tfd = tfidf.transform(texts).toarray().astype("float32")
    return model.predict([seq, tfd], verbose=0)


def run_hybrid_seed(seed):
    print(f"\n========== HYBRID SEED {seed} ==========")
    model = build_hybrid(seed)

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=0
    )

    history = model.fit(
        [seq_train, tfidf_train],
        y_h_train,
        validation_data=([seq_val, tfidf_val], y_h_val),
        epochs=60,
        batch_size=32,
        class_weight=hilbot_class_weights,
        callbacks=[early_stop],
        verbose=0
    )

    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    best_val_loss = float(np.min(history.history["val_loss"]))

    q_frames, m_rows = [], []

    for condition_name, eval_df in CONDITIONS.items():
        probs = hybrid_probs_for_df(model, eval_df)
        q, m = evaluate_probabilities(
            model_name="Hybrid",
            seed=seed,
            condition_name=condition_name,
            eval_df=eval_df,
            probs=probs,
            classes=label_encoder.classes_,
        )
        q_frames.append(q)
        m_rows.append(m)

    qdf = pd.concat(q_frames, ignore_index=True)
    mdf = pd.DataFrame(m_rows)
    train_info = pd.DataFrame([{
        "model": "Hybrid",
        "seed": seed,
        "best_epoch": best_epoch,
        "selection_metric": "val_loss",
        "best_validation_value": best_val_loss,
    }])

    del model
    tf.keras.backend.clear_session()
    gc.collect()

    return qdf, mdf, train_info


print("✓ Hybrid functions ready.")


✓ Hybrid functions ready.


In [ ]:

# ============================================================
# 12. STATIC DISTILBERT INPUTS + HELPERS
# ============================================================

MODEL_NAME = "distilbert/distilbert-base-uncased"
MAX_BERT_LEN = 64
BATCH_SIZE = 16
DISTIL_DEVICE = torch.device("cpu")

distil_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

distil_train_df = train_df.iloc[train_idx].reset_index(drop=True)
distil_val_df = train_df.iloc[val_idx].reset_index(drop=True)

y_distil_train = label_encoder.transform(
    distil_train_df["label"].astype(str)
)
y_distil_val = label_encoder.transform(
    distil_val_df["label"].astype(str)
)

bert_class_weights_np = np.array(
    [hilbot_class_weights[i] for i in range(NUM_CLASSES)],
    dtype=np.float32
)


def tokenize_distil(texts):
    return distil_tokenizer(
        list(pd.Series(texts).astype(str)),
        padding="max_length",
        truncation=True,
        max_length=MAX_BERT_LEN,
        return_tensors="pt"
    )


train_enc = tokenize_distil(distil_train_df["text"])
val_enc = tokenize_distil(distil_val_df["text"])

train_dataset_base = TensorDataset(
    train_enc["input_ids"],
    train_enc["attention_mask"],
    torch.tensor(y_distil_train, dtype=torch.long)
)
val_dataset_base = TensorDataset(
    val_enc["input_ids"],
    val_enc["attention_mask"],
    torch.tensor(y_distil_val, dtype=torch.long)
)


def make_distil_loaders(seed):
    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset_base,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )
    val_loader = DataLoader(
        val_dataset_base,
        batch_size=BATCH_SIZE,
        shuffle=False
    )
    return train_loader, val_loader


def evaluate_bert_loader(model, loader, loss_fn):
    model.eval()
    all_true, all_pred = [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch[0].to(DISTIL_DEVICE)
            attention_mask = batch[1].to(DISTIL_DEVICE)
            labels = batch[2].to(DISTIL_DEVICE)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            ).logits

            total_loss += loss_fn(logits, labels).item()
            pred = torch.argmax(logits, dim=1)

            all_true.extend(labels.cpu().numpy())
            all_pred.extend(pred.cpu().numpy())

    return {
        "loss": total_loss / len(loader),
        "accuracy": accuracy_score(all_true, all_pred),
        "macro_f1": f1_score(
            all_true, all_pred, average="macro", zero_division=0
        )
    }


def distil_probs_for_df(model, eval_df):
    enc = tokenize_distil(eval_df["text"])
    dataset = TensorDataset(enc["input_ids"], enc["attention_mask"])
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    logits_all = []

    model.eval()
    with torch.no_grad():
        for batch in loader:
            logits = model(
                input_ids=batch[0].to(DISTIL_DEVICE),
                attention_mask=batch[1].to(DISTIL_DEVICE)
            ).logits
            logits_all.append(logits.cpu())

    logits = torch.cat(logits_all, dim=0)
    return torch.softmax(logits, dim=1).numpy()


print("DistilBERT device:", DISTIL_DEVICE)
print("✓ DistilBERT static inputs ready.")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

DistilBERT device: cpu
✓ DistilBERT static inputs ready.


In [ ]:

# ============================================================
# 13. DISTILBERT TRAIN / EVALUATE
# ============================================================

def run_distilbert_seed(seed):
    print(f"\n========== DISTILBERT SEED {seed} ==========")

    set_all_seeds(seed)
    train_loader, val_loader = make_distil_loaders(seed)

    id2label = {
        i: label for i, label in enumerate(label_encoder.classes_)
    }
    label2id = {
        label: i for i, label in id2label.items()
    }

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_CLASSES,
        id2label=id2label,
        label2id=label2id
    )
    model.to(DISTIL_DEVICE)

    class_weights = torch.tensor(
        bert_class_weights_np,
        dtype=torch.float32,
        device=DISTIL_DEVICE
    )
    loss_fn = CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=2e-5,
        weight_decay=0.01
    )

    MAX_EPOCHS = 5
    PATIENCE = 3

    best_val_macro_f1 = -np.inf
    best_epoch = None
    best_state = None
    patience_counter = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss = 0.0

        for batch in train_loader:
            input_ids = batch[0].to(DISTIL_DEVICE)
            attention_mask = batch[1].to(DISTIL_DEVICE)
            labels = batch[2].to(DISTIL_DEVICE)

            optimizer.zero_grad()

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            ).logits

            loss = loss_fn(logits, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        val_metrics = evaluate_bert_loader(model, val_loader, loss_fn)

        print(
            f"epoch={epoch} "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_acc={val_metrics['accuracy']:.4f} "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.to(DISTIL_DEVICE)
    model.eval()

    q_frames, m_rows = [], []

    for condition_name, eval_df in CONDITIONS.items():
        probs = distil_probs_for_df(model, eval_df)

        q, m = evaluate_probabilities(
            model_name="DistilBERT",
            seed=seed,
            condition_name=condition_name,
            eval_df=eval_df,
            probs=probs,
            classes=label_encoder.classes_,
        )
        q_frames.append(q)
        m_rows.append(m)

    qdf = pd.concat(q_frames, ignore_index=True)
    mdf = pd.DataFrame(m_rows)
    train_info = pd.DataFrame([{
        "model": "DistilBERT",
        "seed": seed,
        "best_epoch": best_epoch,
        "selection_metric": "val_macro_f1",
        "best_validation_value": best_val_macro_f1,
    }])

    del model, best_state
    gc.collect()

    return qdf, mdf, train_info


print("✓ DistilBERT functions ready.")


✓ DistilBERT functions ready.


In [ ]:

# ============================================================
# 14. RUN / RESUME 3-SEED NEURAL VALIDATION
# ============================================================

FORCE_RERUN = False

def safe_tag(model_name):
    return (
        model_name.lower()
        .replace(" ", "_")
        .replace("+", "plus")
        .replace("-", "_")
    )


all_neural_queries = []
all_neural_metrics = []
all_training_info = []

for model_name, runner in [
    ("Hybrid", run_hybrid_seed),
    ("DistilBERT", run_distilbert_seed),
]:
    for seed in NEURAL_SEEDS:
        tag = safe_tag(model_name)

        q_path = OUT_DIR / f"{tag}_seed{seed}_queries.csv"
        m_path = OUT_DIR / f"{tag}_seed{seed}_metrics.csv"
        t_path = OUT_DIR / f"{tag}_seed{seed}_training.csv"

        if (
            (not FORCE_RERUN)
            and q_path.exists()
            and m_path.exists()
            and t_path.exists()
        ):
            print(f"Loading checkpoint: {model_name} seed {seed}")
            qdf = pd.read_csv(q_path)
            mdf = pd.read_csv(m_path)
            tdf = pd.read_csv(t_path)

        else:
            qdf, mdf, tdf = runner(seed)
            qdf.to_csv(q_path, index=False)
            mdf.to_csv(m_path, index=False)
            tdf.to_csv(t_path, index=False)
            print(f"Saved checkpoint: {model_name} seed {seed}")

        all_neural_queries.append(qdf)
        all_neural_metrics.append(mdf)
        all_training_info.append(tdf)

neural_queries = pd.concat(all_neural_queries, ignore_index=True)
neural_metrics = pd.concat(all_neural_metrics, ignore_index=True)
training_info = pd.concat(all_training_info, ignore_index=True)

neural_queries.to_csv(OUT_DIR / "neural_all_query_results.csv", index=False)
neural_metrics.to_csv(OUT_DIR / "neural_all_condition_metrics.csv", index=False)
training_info.to_csv(OUT_DIR / "neural_training_summary.csv", index=False)

print("\nTRAINING SELECTION")
display(training_info)

print("\nCLEAN NEURAL RESULTS BY SEED")
display(
    neural_metrics[
        neural_metrics["condition"] == "clean_full"
    ][[
        "model", "seed", "accuracy", "macro_f1", "weighted_f1",
        "ece", "brier", "nll",
        "auroc_confidence", "auroc_margin", "auroc_entropy",
        "aurc_confidence", "aurc_margin", "aurc_entropy"
    ]].round(4)
)



========== HYBRID SEED 42 ==========
Saved checkpoint: Hybrid seed 42

========== HYBRID SEED 123 ==========
Saved checkpoint: Hybrid seed 123

========== HYBRID SEED 2026 ==========
Saved checkpoint: Hybrid seed 2026

========== DISTILBERT SEED 42 ==========


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch=1 train_loss=3.4197 val_loss=3.2319 val_acc=0.6721 val_macro_f1=0.2332
epoch=2 train_loss=3.0182 val_loss=2.7087 val_acc=0.7760 val_macro_f1=0.4493
epoch=3 train_loss=2.4940 val_loss=2.1976 val_acc=0.8525 val_macro_f1=0.6896
epoch=4 train_loss=1.9906 val_loss=1.7234 val_acc=0.8962 val_macro_f1=0.8029
epoch=5 train_loss=1.4970 val_loss=1.2801 val_acc=0.9016 val_macro_f1=0.8108
Saved checkpoint: DistilBERT seed 2026

TRAINING SELECTION


,model,seed,best_epoch,selection_metric,best_validation_value
0,Hybrid,42,25,val_loss,0.208759
1,Hybrid,123,30,val_loss,0.162258
2,Hybrid,2026,21,val_loss,0.258727
3,DistilBERT,42,5,val_macro_f1,0.784126
4,DistilBERT,123,4,val_macro_f1,0.848917
5,DistilBERT,2026,5,val_macro_f1,0.810822



CLEAN NEURAL RESULTS BY SEED


,model,seed,accuracy,macro_f1,weighted_f1,ece,brier,nll,auroc_confidence,auroc_margin,auroc_entropy,aurc_confidence,aurc_margin,aurc_entropy
0,Hybrid,42,0.9148,0.8358,0.9173,0.0330,0.1296,0.3272,0.9468,0.9487,0.9440,0.0084,0.0082,0.0086
7,Hybrid,123,0.9016,0.8065,0.9034,0.0450,0.1457,0.3822,0.9510,0.9501,0.9498,0.0099,0.0100,0.0100
14,Hybrid,2026,0.8951,0.7964,0.8983,0.0535,0.1515,0.3541,0.9525,0.9507,0.9512,0.0107,0.0109,0.0109
21,DistilBERT,42,0.8984,0.8645,0.9086,0.4021,0.3344,0.9684,0.9654,0.9553,0.9628,0.0091,0.0102,0.0094
28,DistilBERT,123,0.8918,0.8413,0.9044,0.4608,0.4063,1.1754,0.9698,0.9474,0.9592,0.0094,0.0119,0.0106
35,DistilBERT,2026,0.8852,0.8088,0.8917,0.3446,0.3067,0.9266,0.9590,0.9546,0.9447,0.0118,0.0123,0.0139


In [ ]:

# ============================================================
# 15. MULTI-SEED SUMMARY + SEED-42 CONSISTENCY CHECK
# ============================================================

metrics_to_summarize = [
    "accuracy", "macro_f1", "weighted_f1",
    "ece", "brier", "nll",
    "auroc_confidence", "auroc_margin", "auroc_entropy",
    "aurc_confidence", "aurc_margin", "aurc_entropy",
]

clean_neural = neural_metrics[
    neural_metrics["condition"] == "clean_full"
].copy()

summary_rows = []

for model_name, g in clean_neural.groupby("model"):
    row = {"model": model_name, "n_seeds": g["seed"].nunique()}

    for metric in metrics_to_summarize:
        row[f"{metric}_mean"] = g[metric].mean()
        row[f"{metric}_sd"] = g[metric].std(ddof=1)
        row[f"{metric}_min"] = g[metric].min()
        row[f"{metric}_max"] = g[metric].max()

    summary_rows.append(row)

neural_clean_seed_summary = pd.DataFrame(summary_rows)
neural_clean_seed_summary.to_csv(
    OUT_DIR / "neural_clean_multiseed_summary.csv",
    index=False
)

print("MULTI-SEED CLEAN SUMMARY")
display(neural_clean_seed_summary.round(4))


# Per-seed perturbation drops
drop_rows = []

for model_name in ["Hybrid", "DistilBERT"]:
    for seed in NEURAL_SEEDS:
        m = neural_metrics[
            (neural_metrics["model"] == model_name)
            & (neural_metrics["seed"] == seed)
        ].set_index("condition")

        for perturbation, (clean_c, pert_c) in PAIRS.items():
            drop_rows.append({
                "model": model_name,
                "seed": seed,
                "perturbation": perturbation,
                "accuracy_drop": (
                    m.loc[clean_c, "accuracy"]
                    - m.loc[pert_c, "accuracy"]
                ),
                "margin_auroc_clean": m.loc[clean_c, "auroc_margin"],
                "margin_auroc_perturbed": m.loc[pert_c, "auroc_margin"],
                "margin_aurc_clean": m.loc[clean_c, "aurc_margin"],
                "margin_aurc_perturbed": m.loc[pert_c, "aurc_margin"],
            })

neural_seed_drops = pd.DataFrame(drop_rows)

neural_drop_summary = (
    neural_seed_drops
    .groupby(["model", "perturbation"], as_index=False)
    .agg(
        n_seeds=("seed", "nunique"),
        drop_mean=("accuracy_drop", "mean"),
        drop_sd=("accuracy_drop", "std"),
        drop_min=("accuracy_drop", "min"),
        drop_max=("accuracy_drop", "max"),
        pert_margin_auroc_mean=("margin_auroc_perturbed", "mean"),
        pert_margin_auroc_sd=("margin_auroc_perturbed", "std"),
        pert_margin_aurc_mean=("margin_aurc_perturbed", "mean"),
        pert_margin_aurc_sd=("margin_aurc_perturbed", "std"),
    )
)

neural_seed_drops.to_csv(
    OUT_DIR / "neural_per_seed_robustness.csv",
    index=False
)
neural_drop_summary.to_csv(
    OUT_DIR / "neural_multiseed_robustness_summary.csv",
    index=False
)

print("\nMULTI-SEED ROBUSTNESS SUMMARY")
display(neural_drop_summary.round(4))


# ------------------------------------------------------------
# Seed-42 comparison with the recorded Book-1 headline values.
# This is a WARNING check, not an assertion, because exact numeric
# identity can change with library/runtime/hardware details.
# ------------------------------------------------------------

BOOK1_EXPECTED = pd.DataFrame([
    {
        "model": "Hybrid",
        "accuracy": 0.9148,
        "macro_f1": 0.8358,
        "weighted_f1": 0.9173,
        "ece": 0.0330,
        "brier": 0.1296,
        "nll": 0.3272,
        "auroc_margin": 0.9487,
        "aurc_margin": 0.0082,
    },
    {
        "model": "DistilBERT",
        "accuracy": 0.8820,
        "macro_f1": 0.7955,
        "weighted_f1": 0.8862,
        "ece": 0.5109,
        "brier": 0.4599,
        "nll": 1.2668,
        "auroc_margin": 0.9553,
        "aurc_margin": 0.0129,
    },
])

seed42 = neural_metrics[
    (neural_metrics["seed"] == 42)
    & (neural_metrics["condition"] == "clean_full")
].copy()

check_rows = []

for _, expected in BOOK1_EXPECTED.iterrows():
    observed = seed42[seed42["model"] == expected["model"]].iloc[0]

    row = {"model": expected["model"]}

    for metric in [
        "accuracy", "macro_f1", "weighted_f1",
        "ece", "brier", "nll",
        "auroc_margin", "aurc_margin"
    ]:
        row[f"{metric}_book1"] = expected[metric]
        row[f"{metric}_book3"] = observed[metric]
        row[f"{metric}_diff"] = observed[metric] - expected[metric]

    check_rows.append(row)

seed42_consistency = pd.DataFrame(check_rows)
seed42_consistency.to_csv(
    OUT_DIR / "seed42_book1_consistency_check.csv",
    index=False
)

print("\nSEED-42 BOOK-1 CONSISTENCY CHECK")
display(seed42_consistency.round(4))

print(
    "\nInterpretation: small differences can reflect runtime/library nondeterminism. "
    "Large deviations should be investigated before updating the paper."
)


MULTI-SEED CLEAN SUMMARY


,model,n_seeds,accuracy_mean,accuracy_sd,accuracy_min,accuracy_max,macro_f1_mean,macro_f1_sd,macro_f1_min,macro_f1_max,...,aurc_confidence_min,aurc_confidence_max,aurc_margin_mean,aurc_margin_sd,aurc_margin_min,aurc_margin_max,aurc_entropy_mean,aurc_entropy_sd,aurc_entropy_min,aurc_entropy_max
0,DistilBERT,3,0.8918,0.0066,0.8852,0.8984,0.8382,0.0280,0.8088,0.8645,...,0.0091,0.0118,0.0115,0.0011,0.0102,0.0123,0.0113,0.0023,0.0094,0.0139
1,Hybrid,3,0.9038,0.0100,0.8951,0.9148,0.8129,0.0205,0.7964,0.8358,...,0.0084,0.0107,0.0097,0.0014,0.0082,0.0109,0.0098,0.0011,0.0086,0.0109



MULTI-SEED ROBUSTNESS SUMMARY


,model,perturbation,n_seeds,drop_mean,drop_sd,drop_min,drop_max,pert_margin_auroc_mean,pert_margin_auroc_sd,pert_margin_aurc_mean,pert_margin_aurc_sd
0,DistilBERT,abbreviation,3,0.1572,0.0303,0.1226,0.1792,0.8241,0.0051,0.1957,0.0253
1,DistilBERT,shortening,3,0.0460,0.0100,0.0345,0.0517,0.8350,0.0315,0.1266,0.0212
2,DistilBERT,typo,3,0.1881,0.0288,0.1683,0.2211,0.8693,0.0223,0.0935,0.0251
3,Hybrid,abbreviation,3,0.1635,0.0237,0.1415,0.1887,0.8353,0.0096,0.1607,0.0028
4,Hybrid,shortening,3,0.0603,0.0172,0.0431,0.0776,0.8506,0.0180,0.0977,0.0036
5,Hybrid,typo,3,0.1232,0.0125,0.1089,0.1320,0.9075,0.0094,0.0467,0.0013



SEED-42 BOOK-1 CONSISTENCY CHECK


,model,accuracy_book1,accuracy_book3,accuracy_diff,macro_f1_book1,macro_f1_book3,macro_f1_diff,weighted_f1_book1,weighted_f1_book3,weighted_f1_diff,...,brier_diff,nll_book1,nll_book3,nll_diff,auroc_margin_book1,auroc_margin_book3,auroc_margin_diff,aurc_margin_book1,aurc_margin_book3,aurc_margin_diff
0,Hybrid,0.9148,0.9148,-0.0000,0.8358,0.8358,-0.000,0.9173,0.9173,0.0000,...,-0.0000,0.3272,0.3272,0.0000,0.9487,0.9487,0.0,0.0082,0.0082,0.0000
1,DistilBERT,0.8820,0.8984,0.0164,0.7955,0.8645,0.069,0.8862,0.9086,0.0224,...,-0.1255,1.2668,0.9684,-0.2984,0.9553,0.9553,-0.0,0.0129,0.0102,-0.0027



Interpretation: small differences can reflect runtime/library nondeterminism. Large deviations should be investigated before updating the paper.



## Stage D — Statistical hardening

The next cells use **canonical seed 42** for the same query-level paired tests as the current paper, then correct multiplicity. Multi-seed stability is reported separately rather than treating different seeds as independent test examples.


In [ ]:

# ============================================================
# 16. HOLM-CORRECTED PAIRED TESTS
# ============================================================

all_queries = pd.concat(
    [classical_queries, neural_queries],
    ignore_index=True
)

CANONICAL_MODELS = [
    "LR",
    "Calibrated SVM",
    "DistilBERT",
    "Hybrid",
]

def get_query_block(model_name, seed, condition):
    q = all_queries[
        (all_queries["model"] == model_name)
        & (all_queries["seed"] == seed)
        & (all_queries["condition"] == condition)
    ].sort_values("row_id").reset_index(drop=True)
    return q


paired_rows = []
uncertainty_rows = []

for model_name in CANONICAL_MODELS:
    seed = 42

    for perturbation, (clean_c, pert_c) in PAIRS.items():
        clean_q = get_query_block(model_name, seed, clean_c)
        pert_q = get_query_block(model_name, seed, pert_c)

        assert len(clean_q) == len(pert_q)
        assert np.array_equal(clean_q["true_label"], pert_q["true_label"])

        c = clean_q["correct"].astype(bool).to_numpy()
        p = pert_q["correct"].astype(bool).to_numpy()

        harmed = int(np.sum(c & ~p))
        helped = int(np.sum(~c & p))
        discordant = harmed + helped

        raw_p = (
            binomtest(
                min(harmed, helped),
                n=discordant,
                p=0.5,
                alternative="two-sided"
            ).pvalue
            if discordant > 0
            else 1.0
        )

        paired_rows.append({
            "model": model_name,
            "seed": seed,
            "perturbation": perturbation,
            "n": len(c),
            "both_correct": int(np.sum(c & p)),
            "harmed_correct_to_wrong": harmed,
            "helped_wrong_to_correct": helped,
            "both_wrong": int(np.sum(~c & ~p)),
            "accuracy_change": float(p.mean() - c.mean()),
            "accuracy_drop": float(c.mean() - p.mean()),
            "mcnemar_raw_p": raw_p,
        })

        for metric in ["confidence", "margin", "entropy"]:
            wp, delta = safe_wilcoxon_delta(
                clean_q[metric].to_numpy(),
                pert_q[metric].to_numpy()
            )
            uncertainty_rows.append({
                "model": model_name,
                "seed": seed,
                "perturbation": perturbation,
                "metric": metric,
                "mean_change_pert_minus_clean": delta,
                "wilcoxon_raw_p": wp,
            })

paired_tests = pd.DataFrame(paired_rows)
paired_tests["mcnemar_holm_p"] = holm_adjust(
    paired_tests["mcnemar_raw_p"].values
)
paired_tests["significant_holm_0.05"] = (
    paired_tests["mcnemar_holm_p"] < 0.05
)

uncertainty_tests = pd.DataFrame(uncertainty_rows)
uncertainty_tests["wilcoxon_holm_all36_p"] = holm_adjust(
    uncertainty_tests["wilcoxon_raw_p"].fillna(1.0).values
)
uncertainty_tests["significant_holm_all36_0.05"] = (
    uncertainty_tests["wilcoxon_holm_all36_p"] < 0.05
)

paired_tests.to_csv(
    OUT_DIR / "holm_corrected_accuracy_tests.csv",
    index=False
)
uncertainty_tests.to_csv(
    OUT_DIR / "holm_corrected_uncertainty_tests.csv",
    index=False
)

print("HOLM-CORRECTED PRIMARY ACCURACY TESTS")
display(
    paired_tests[[
        "model", "perturbation", "n",
        "harmed_correct_to_wrong", "helped_wrong_to_correct",
        "accuracy_drop", "mcnemar_raw_p",
        "mcnemar_holm_p", "significant_holm_0.05"
    ]].round(6)
)

print("\nSECONDARY UNCERTAINTY TESTS — HOLM ACROSS ALL 36")
display(
    uncertainty_tests.round(6)
)


HOLM-CORRECTED PRIMARY ACCURACY TESTS


,model,perturbation,n,harmed_correct_to_wrong,helped_wrong_to_correct,accuracy_drop,mcnemar_raw_p,mcnemar_holm_p,significant_holm_0.05
0,LR,typo,303,19,8,0.036304,0.052239,0.261195,False
1,LR,abbreviation,106,12,5,0.066038,0.143463,0.573853,False
2,LR,shortening,116,7,12,-0.043103,0.359283,0.867188,False
3,Calibrated SVM,typo,303,28,8,0.066007,0.001193,0.009546,True
4,Calibrated SVM,abbreviation,106,15,4,0.103774,0.019211,0.134476,False
5,Calibrated SVM,shortening,116,14,14,0.000000,1.000000,1.000000,False
6,DistilBERT,typo,303,53,0,0.174917,0.000000,0.000000,True
7,DistilBERT,abbreviation,106,21,2,0.179245,0.000066,0.000594,True
8,DistilBERT,shortening,116,6,2,0.034483,0.289062,0.867188,False
9,Hybrid,typo,303,41,2,0.128713,0.000000,0.000000,True



SECONDARY UNCERTAINTY TESTS — HOLM ACROSS ALL 36


,model,seed,perturbation,metric,mean_change_pert_minus_clean,wilcoxon_raw_p,wilcoxon_holm_all36_p,significant_holm_all36_0.05
0,LR,42,typo,confidence,-0.041284,0.000000,0.000000,True
1,LR,42,typo,margin,-0.042736,0.000000,0.000000,True
2,LR,42,typo,entropy,0.087386,0.000000,0.000000,True
3,LR,42,abbreviation,confidence,-0.040030,0.000529,0.007409,True
4,LR,42,abbreviation,margin,-0.026460,0.004190,0.041900,True
5,LR,42,abbreviation,entropy,0.107919,0.002881,0.031696,True
6,LR,42,shortening,confidence,-0.005014,0.555518,1.000000,False
7,LR,42,shortening,margin,-0.013070,0.141878,0.793038,False
8,LR,42,shortening,entropy,-0.022626,0.175183,0.793038,False
9,Calibrated SVM,42,typo,confidence,-0.084166,0.000000,0.000000,True


In [ ]:

# ============================================================
# 17. DIRECT BETWEEN-MODEL ROBUSTNESS TEST
# Paired bootstrap CI for difference in degradation.
# Positive difference => neural model degrades MORE than SVM.
# ============================================================

BOOTSTRAP_B = 10000
BOOTSTRAP_SEED = 42

def degradation_vector(model_name, seed, clean_condition, pert_condition):
    clean_q = get_query_block(
        model_name, seed, clean_condition
    ).sort_values("row_id")
    pert_q = get_query_block(
        model_name, seed, pert_condition
    ).sort_values("row_id")

    assert len(clean_q) == len(pert_q)
    assert np.array_equal(clean_q["true_label"], pert_q["true_label"])

    # +1 = harmed, 0 = unchanged, -1 = helped
    return (
        clean_q["correct"].astype(int).to_numpy()
        - pert_q["correct"].astype(int).to_numpy()
    )


def paired_bootstrap_drop_difference(
    model_a,
    model_b,
    seed,
    clean_condition,
    pert_condition,
    B=10000,
    rng_seed=42
):
    a = degradation_vector(
        model_a, seed, clean_condition, pert_condition
    )
    b = degradation_vector(
        model_b, seed, clean_condition, pert_condition
    )

    assert len(a) == len(b)
    observed = float(a.mean() - b.mean())

    rng = np.random.default_rng(rng_seed)
    boot = np.empty(B, dtype=float)
    n = len(a)

    for i in range(B):
        idx = rng.integers(0, n, size=n)
        boot[i] = a[idx].mean() - b[idx].mean()

    lo, hi = np.quantile(boot, [0.025, 0.975])

    return {
        "model_a": model_a,
        "model_b": model_b,
        "seed": seed,
        "n": n,
        "drop_a": float(a.mean()),
        "drop_b": float(b.mean()),
        "drop_difference_a_minus_b": observed,
        "ci95_low": float(lo),
        "ci95_high": float(hi),
        "ci_excludes_zero": bool((lo > 0) or (hi < 0)),
        "bootstrap_B": B,
    }


bootstrap_rows = []

for perturbation, (clean_c, pert_c) in PAIRS.items():
    for neural_model in ["Hybrid", "DistilBERT"]:
        row = paired_bootstrap_drop_difference(
            model_a=neural_model,
            model_b="Calibrated SVM",
            seed=42,
            clean_condition=clean_c,
            pert_condition=pert_c,
            B=BOOTSTRAP_B,
            rng_seed=BOOTSTRAP_SEED,
        )
        row["perturbation"] = perturbation
        bootstrap_rows.append(row)

bootstrap_degradation = pd.DataFrame(bootstrap_rows)

bootstrap_degradation.to_csv(
    OUT_DIR / "paired_bootstrap_degradation_differences.csv",
    index=False
)

print(
    "Positive difference means the neural model lost more accuracy "
    "than calibrated SVM on the same paired queries."
)
display(bootstrap_degradation.round(4))


Positive difference means the neural model lost more accuracy than calibrated SVM on the same paired queries.


,model_a,model_b,seed,n,drop_a,drop_b,drop_difference_a_minus_b,ci95_low,ci95_high,ci_excludes_zero,bootstrap_B,perturbation
0,Hybrid,Calibrated SVM,42,303,0.1287,0.0660,0.0627,0.0231,0.1056,True,10000,typo
1,DistilBERT,Calibrated SVM,42,303,0.1749,0.0660,0.1089,0.0561,0.1650,True,10000,typo
2,Hybrid,Calibrated SVM,42,106,0.1887,0.1038,0.0849,0.0094,0.1604,True,10000,abbreviation
3,DistilBERT,Calibrated SVM,42,106,0.1792,0.1038,0.0755,-0.0283,0.1792,False,10000,abbreviation
4,Hybrid,Calibrated SVM,42,116,0.0776,0.0000,0.0776,-0.0259,0.1810,False,10000,shortening
5,DistilBERT,Calibrated SVM,42,116,0.0345,0.0000,0.0345,-0.0690,0.1379,False,10000,shortening


In [ ]:

# ============================================================
# 18. CALIBRATION UNDER PERTURBATION
# ============================================================

all_metrics = pd.concat(
    [classical_metrics, neural_metrics],
    ignore_index=True
)

calibration_rows = []

for model_name in CANONICAL_MODELS:
    seeds = (
        [42]
        if model_name in ["LR", "Calibrated SVM"]
        else NEURAL_SEEDS
    )

    for seed in seeds:
        m = all_metrics[
            (all_metrics["model"] == model_name)
            & (all_metrics["seed"] == seed)
        ].set_index("condition")

        for perturbation, (clean_c, pert_c) in PAIRS.items():
            calibration_rows.append({
                "model": model_name,
                "seed": seed,
                "perturbation": perturbation,
                "n": int(m.loc[pert_c, "n"]),
                "ece_clean": m.loc[clean_c, "ece"],
                "ece_perturbed": m.loc[pert_c, "ece"],
                "ece_change": m.loc[pert_c, "ece"] - m.loc[clean_c, "ece"],
                "brier_clean": m.loc[clean_c, "brier"],
                "brier_perturbed": m.loc[pert_c, "brier"],
                "brier_change": m.loc[pert_c, "brier"] - m.loc[clean_c, "brier"],
                "nll_clean": m.loc[clean_c, "nll"],
                "nll_perturbed": m.loc[pert_c, "nll"],
                "nll_change": m.loc[pert_c, "nll"] - m.loc[clean_c, "nll"],
            })

calibration_shift = pd.DataFrame(calibration_rows)

calibration_shift_summary = (
    calibration_shift
    .groupby(["model", "perturbation"], as_index=False)
    .agg(
        n_runs=("seed", "nunique"),
        ece_change_mean=("ece_change", "mean"),
        ece_change_sd=("ece_change", "std"),
        brier_change_mean=("brier_change", "mean"),
        brier_change_sd=("brier_change", "std"),
        nll_change_mean=("nll_change", "mean"),
        nll_change_sd=("nll_change", "std"),
    )
)

calibration_shift.to_csv(
    OUT_DIR / "calibration_shift_per_run.csv",
    index=False
)
calibration_shift_summary.to_csv(
    OUT_DIR / "calibration_shift_summary.csv",
    index=False
)

print("CALIBRATION SHIFT — PERTURBED MINUS MATCHED CLEAN")
display(calibration_shift_summary.round(4))


CALIBRATION SHIFT — PERTURBED MINUS MATCHED CLEAN


,model,perturbation,n_runs,ece_change_mean,ece_change_sd,brier_change_mean,brier_change_sd,nll_change_mean,nll_change_sd
0,Calibrated SVM,abbreviation,1,0.0130,NaN,0.0765,NaN,0.4400,NaN
1,Calibrated SVM,shortening,1,0.0268,NaN,0.0401,NaN,0.1185,NaN
2,Calibrated SVM,typo,1,0.0353,NaN,0.0957,NaN,0.3146,NaN
3,DistilBERT,abbreviation,3,-0.1043,0.0146,0.0765,0.0240,0.2457,0.0655
4,DistilBERT,shortening,3,-0.0268,0.0168,0.0344,0.0071,0.0574,0.0142
5,DistilBERT,typo,3,-0.0944,0.0392,0.1616,0.0079,0.3830,0.0168
6,Hybrid,abbreviation,3,0.0390,0.0395,0.1967,0.0373,0.6185,0.1338
7,Hybrid,shortening,3,-0.0251,0.0344,0.0613,0.0370,0.1801,0.0930
8,Hybrid,typo,3,0.0585,0.0241,0.1724,0.0152,0.3991,0.0352
9,LR,abbreviation,1,-0.0493,NaN,0.0368,NaN,0.2391,NaN


In [ ]:

# ============================================================
# 19. SOURCE-STRATIFIED SANITY CHECK
# ============================================================

def source_metrics_from_queries(q):
    rows = []

    for (
        model_name,
        seed,
        condition,
        source
    ), g in q.groupby(
        ["model", "seed", "condition", "source"],
        dropna=False
    ):
        correct = g["correct"].astype(bool).to_numpy()

        row = {
            "model": model_name,
            "seed": seed,
            "condition": condition,
            "source": source,
            "n": len(g),
            "accuracy": correct.mean(),
            "mean_margin": g["margin"].mean(),
        }

        if len(np.unique(correct)) == 2:
            row["margin_auroc"] = roc_auc_score(
                correct.astype(int),
                g["margin"].to_numpy()
            )
            row["margin_aurc"] = aurc_from_score(
                correct,
                g["margin"].to_numpy()
            )
        else:
            row["margin_auroc"] = np.nan
            row["margin_aurc"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows)


source_audit = source_metrics_from_queries(all_queries)

# Focused display: full clean + typo.
focused_source_audit = source_audit[
    source_audit["condition"].isin(["clean_full", "clean_typo", "typo"])
].copy()

source_summary = (
    focused_source_audit
    .groupby(["model", "condition", "source"], as_index=False)
    .agg(
        n_runs=("seed", "nunique"),
        n_mean=("n", "mean"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_sd=("accuracy", "std"),
        margin_auroc_mean=("margin_auroc", "mean"),
        margin_auroc_sd=("margin_auroc", "std"),
        margin_aurc_mean=("margin_aurc", "mean"),
        margin_aurc_sd=("margin_aurc", "std"),
    )
)

source_audit.to_csv(
    OUT_DIR / "source_stratified_per_run.csv",
    index=False
)
source_summary.to_csv(
    OUT_DIR / "source_stratified_summary.csv",
    index=False
)

print(
    "Use this as a sanity check, NOT as proof that one source is inherently harder: "
    "CSV and JSON sources have different intent distributions."
)
display(source_summary.round(4))


Use this as a sanity check, NOT as proof that one source is inherently harder: CSV and JSON sources have different intent distributions.


,model,condition,source,n_runs,n_mean,accuracy_mean,accuracy_sd,margin_auroc_mean,margin_auroc_sd,margin_aurc_mean,margin_aurc_sd
0,Calibrated SVM,clean_full,csv,1,180.0,1.0000,NaN,NaN,NaN,NaN,NaN
1,Calibrated SVM,clean_full,json,1,125.0,0.5520,NaN,0.7270,NaN,0.2550,NaN
2,Calibrated SVM,clean_typo,csv,1,179.0,1.0000,NaN,NaN,NaN,NaN,NaN
3,Calibrated SVM,clean_typo,json,1,124.0,0.5484,NaN,0.7230,NaN,0.2609,NaN
4,Calibrated SVM,typo,csv,1,179.0,0.9665,NaN,0.9894,NaN,0.0009,NaN
5,Calibrated SVM,typo,json,1,124.0,0.4355,NaN,0.7357,NaN,0.3428,NaN
6,DistilBERT,clean_full,csv,3,180.0,1.0000,0.0000,NaN,NaN,NaN,NaN
7,DistilBERT,clean_full,json,3,125.0,0.7360,0.0160,0.8593,0.0131,0.0832,0.0064
8,DistilBERT,clean_typo,csv,3,179.0,1.0000,0.0000,NaN,NaN,NaN,NaN
9,DistilBERT,clean_typo,json,3,124.0,0.7339,0.0161,0.8581,0.0135,0.0846,0.0065



## Stage E — Manual perturbation audit

Open `perturbation_semantic_audit_TO_REVIEW.csv` from the Book-3 output folder and fill:

- `1` = intent clearly preserved
- `0` = intent not preserved
- `2` = uncertain / arguable

Do **not** use the classifier prediction to decide validity. Judge whether a reasonable human would retain the same gold intent after the transformation.

After saving the reviewed CSV back to the same path, run the next cell.


In [ ]:

# ============================================================
# 20. SUMMARIZE COMPLETED MANUAL SEMANTIC AUDIT
# ============================================================

if not SEMANTIC_AUDIT_PATH.exists():
    print("Audit file not found:", SEMANTIC_AUDIT_PATH)
else:
    reviewed = pd.read_csv(SEMANTIC_AUDIT_PATH)

    if "valid_semantics" not in reviewed.columns:
        print("Column valid_semantics is missing.")
    else:
        reviewed["valid_semantics_numeric"] = pd.to_numeric(
            reviewed["valid_semantics"],
            errors="coerce"
        )

        n_reviewed = reviewed["valid_semantics_numeric"].notna().sum()

        if n_reviewed == 0:
            print(
                "No semantic-audit labels entered yet. "
                "Fill valid_semantics with 1 / 0 / 2 and rerun."
            )
        else:
            semantic_summary = (
                reviewed
                .dropna(subset=["valid_semantics_numeric"])
                .groupby("condition")
                .agg(
                    reviewed_n=("valid_semantics_numeric", "size"),
                    valid_n=("valid_semantics_numeric", lambda s: (s == 1).sum()),
                    invalid_n=("valid_semantics_numeric", lambda s: (s == 0).sum()),
                    uncertain_n=("valid_semantics_numeric", lambda s: (s == 2).sum()),
                )
                .reset_index()
            )

            semantic_summary["valid_rate"] = (
                semantic_summary["valid_n"]
                / semantic_summary["reviewed_n"]
            )

            display(semantic_summary.round(4))

            semantic_summary.to_csv(
                OUT_DIR / "perturbation_semantic_audit_summary.csv",
                index=False
            )

            invalid_or_uncertain = reviewed[
                reviewed["valid_semantics_numeric"].isin([0, 2])
            ]

            print("\nINVALID / UNCERTAIN ITEMS")
            display(invalid_or_uncertain)


No semantic-audit labels entered yet. Fill valid_semantics with 1 / 0 / 2 and rerun.


In [ ]:

# ============================================================
# 21. PAPER-READY TABLES
# Primary cross-model uncertainty score = TOP-2 MARGIN.
# ============================================================

def mean_sd_string(mean, sd, digits=4):
    if pd.isna(sd):
        return f"{mean:.{digits}f}"
    return f"{mean:.{digits}f} ± {sd:.{digits}f}"


# Clean primary table
paper_clean_rows = []

for model_name in CANONICAL_MODELS:
    if model_name in ["LR", "Calibrated SVM"]:
        g = all_metrics[
            (all_metrics["model"] == model_name)
            & (all_metrics["seed"] == 42)
            & (all_metrics["condition"] == "clean_full")
        ]
        row = g.iloc[0]

        paper_clean_rows.append({
            "Model": model_name,
            "Accuracy": f"{row['accuracy']:.4f}",
            "Macro F1": f"{row['macro_f1']:.4f}",
            "Weighted F1": f"{row['weighted_f1']:.4f}",
            "ECE": f"{row['ece']:.4f}",
            "Brier": f"{row['brier']:.4f}",
            "NLL": f"{row['nll']:.4f}",
            "Margin AUROC": f"{row['auroc_margin']:.4f}",
            "Margin AURC": f"{row['aurc_margin']:.4f}",
        })

    else:
        g = all_metrics[
            (all_metrics["model"] == model_name)
            & (all_metrics["condition"] == "clean_full")
        ]

        paper_clean_rows.append({
            "Model": model_name,
            "Accuracy": mean_sd_string(g["accuracy"].mean(), g["accuracy"].std(ddof=1)),
            "Macro F1": mean_sd_string(g["macro_f1"].mean(), g["macro_f1"].std(ddof=1)),
            "Weighted F1": mean_sd_string(g["weighted_f1"].mean(), g["weighted_f1"].std(ddof=1)),
            "ECE": mean_sd_string(g["ece"].mean(), g["ece"].std(ddof=1)),
            "Brier": mean_sd_string(g["brier"].mean(), g["brier"].std(ddof=1)),
            "NLL": mean_sd_string(g["nll"].mean(), g["nll"].std(ddof=1)),
            "Margin AUROC": mean_sd_string(g["auroc_margin"].mean(), g["auroc_margin"].std(ddof=1)),
            "Margin AURC": mean_sd_string(g["aurc_margin"].mean(), g["aurc_margin"].std(ddof=1)),
        })

paper_clean_table = pd.DataFrame(paper_clean_rows)

print("PAPER-READY CLEAN TABLE")
display(paper_clean_table)


# Robustness table
classical_drop_rows = []

for model_name in ["LR", "Calibrated SVM"]:
    m = all_metrics[
        (all_metrics["model"] == model_name)
        & (all_metrics["seed"] == 42)
    ].set_index("condition")

    for perturbation, (clean_c, pert_c) in PAIRS.items():
        classical_drop_rows.append({
            "model": model_name,
            "perturbation": perturbation,
            "drop_mean": m.loc[clean_c, "accuracy"] - m.loc[pert_c, "accuracy"],
            "drop_sd": np.nan,
        })

paper_drop_numeric = pd.concat([
    pd.DataFrame(classical_drop_rows),
    neural_drop_summary[[
        "model", "perturbation", "drop_mean", "drop_sd"
    ]]
], ignore_index=True)

paper_robustness = (
    paper_drop_numeric
    .assign(
        Drop=lambda d: [
            mean_sd_string(mu, sd)
            for mu, sd in zip(d["drop_mean"], d["drop_sd"])
        ]
    )
    .pivot(index="model", columns="perturbation", values="Drop")
    .reset_index()
    .rename(columns={"model": "Model"})
)

print("\nPAPER-READY ACCURACY-DROP TABLE")
display(paper_robustness)

paper_clean_table.to_csv(
    OUT_DIR / "PAPER_READY_clean_primary_margin.csv",
    index=False
)
paper_robustness.to_csv(
    OUT_DIR / "PAPER_READY_robustness_multiseed.csv",
    index=False
)


PAPER-READY CLEAN TABLE


,Model,Accuracy,Macro F1,Weighted F1,ECE,Brier,NLL,Margin AUROC,Margin AURC
0,LR,0.7770,0.5390,0.7943,0.5547,0.6814,1.7771,0.7950,0.0935
1,Calibrated SVM,0.8164,0.5651,0.8166,0.1263,0.2552,0.6750,0.9243,0.0319
2,DistilBERT,0.8918 ± 0.0066,0.8382 ± 0.0280,0.9015 ± 0.0088,0.4025 ± 0.0581,0.3491 ± 0.0514,1.0235 ± 0.1332,0.9524 ± 0.0044,0.0115 ± 0.0011
3,Hybrid,0.9038 ± 0.0100,0.8129 ± 0.0205,0.9064 ± 0.0098,0.0438 ± 0.0103,0.1422 ± 0.0114,0.3545 ± 0.0275,0.9498 ± 0.0010,0.0097 ± 0.0014



PAPER-READY ACCURACY-DROP TABLE


perturbation,Model,abbreviation,shortening,typo
0,Calibrated SVM,0.1038,0.0000,0.0660
1,DistilBERT,0.1572 ± 0.0303,0.0460 ± 0.0100,0.1881 ± 0.0288
2,Hybrid,0.1635 ± 0.0237,0.0603 ± 0.0172,0.1232 ± 0.0125
3,LR,0.0660,-0.0431,0.0363


In [ ]:

# ============================================================
# 22. FINAL BOOK-3 STATUS REPORT
# ============================================================

required_outputs = [
    "near_duplicate_query_audit.csv",
    "near_duplicate_summary.csv",
    "template_overlap_summary.csv",
    "classical_condition_metrics.csv",
    "neural_all_condition_metrics.csv",
    "neural_clean_multiseed_summary.csv",
    "neural_multiseed_robustness_summary.csv",
    "seed42_book1_consistency_check.csv",
    "holm_corrected_accuracy_tests.csv",
    "holm_corrected_uncertainty_tests.csv",
    "paired_bootstrap_degradation_differences.csv",
    "calibration_shift_summary.csv",
    "source_stratified_summary.csv",
    "PAPER_READY_clean_primary_margin.csv",
    "PAPER_READY_robustness_multiseed.csv",
]

status = pd.DataFrame({
    "file": required_outputs,
    "exists": [(OUT_DIR / f).exists() for f in required_outputs]
})

display(status)

print("\nBOOK 3 OUTPUT DIRECTORY")
print(OUT_DIR)

print("\nFINAL REVIEWER-HARDENING CHECKLIST")
print("1. Inspect near-duplicate/template audit.")
print("2. Confirm seed-42 rerun is reasonably consistent with Book 1.")
print("3. Use 3-seed mean ± SD for neural clean metrics and perturbation drops.")
print("4. Use top-2 margin as the primary cross-model selective score.")
print("5. Use Holm-adjusted p-values for the 12 primary perturbation accuracy tests.")
print("6. Inspect paired-bootstrap CIs for neural-vs-SVM degradation differences.")
print("7. Report calibration changes under perturbation cautiously, especially for small subsets.")
print("8. Inspect CSV/JSON source-stratified results as a sanity check only.")
print("9. Complete the manual semantic audit before calling perturbations label-preserving.")
print("10. Do not change Book 1 or Book 2 unless Book 3 reveals a genuine error.")


,file,exists
0,near_duplicate_query_audit.csv,True
1,near_duplicate_summary.csv,True
2,template_overlap_summary.csv,True
3,classical_condition_metrics.csv,True
4,neural_all_condition_metrics.csv,True
5,neural_clean_multiseed_summary.csv,True
6,neural_multiseed_robustness_summary.csv,True
7,seed42_book1_consistency_check.csv,True
8,holm_corrected_accuracy_tests.csv,True
9,holm_corrected_uncertainty_tests.csv,True



BOOK 3 OUTPUT DIRECTORY
/content/drive/MyDrive/Hilbot chatbot (1)/Hilbot-FI/book3_reviewer_hardening

FINAL REVIEWER-HARDENING CHECKLIST
1. Inspect near-duplicate/template audit.
2. Confirm seed-42 rerun is reasonably consistent with Book 1.
3. Use 3-seed mean ± SD for neural clean metrics and perturbation drops.
4. Use top-2 margin as the primary cross-model selective score.
5. Use Holm-adjusted p-values for the 12 primary perturbation accuracy tests.
6. Inspect paired-bootstrap CIs for neural-vs-SVM degradation differences.
7. Report calibration changes under perturbation cautiously, especially for small subsets.
8. Inspect CSV/JSON source-stratified results as a sanity check only.
9. Complete the manual semantic audit before calling perturbations label-preserving.
10. Do not change Book 1 or Book 2 unless Book 3 reveals a genuine error.


In [24]:
# ============================================================
# SHORTENING AFTER EXCLUDING 3 AMBIGUOUS PAIRS
# ============================================================

from pathlib import Path
import pandas as pd

reviewed_path = (
    OUT_DIR /
    "perturbation_semantic_audit_REVIEWED_FINAL.csv"
)

print("File exists:", reviewed_path.exists())
print("Reading:", reviewed_path)

reviewed = pd.read_csv(reviewed_path)

print("\nColumns:")
print(reviewed.columns.tolist())

print("\nvalid_semantics values:")
print(reviewed["valid_semantics"].value_counts(dropna=False))

print("\nBy condition:")
print(
    reviewed.groupby(
        ["condition", "valid_semantics"]
    ).size()
)

File exists: True
Reading: /content/drive/MyDrive/Hilbot chatbot (1)/Hilbot-FI/book3_reviewer_hardening/perturbation_semantic_audit_REVIEWED_FINAL.csv

Columns:
['condition', 'original_text', 'perturbed_text', 'label', 'source', 'valid_semantics', 'review_notes']

valid_semantics values:
valid_semantics
1    319
2      3
Name: count, dtype: int64

By condition:
condition     valid_semantics
abbreviation  1                  106
shortening    1                  113
              2                    3
typo          1                  100
dtype: int64


In [25]:
# ============================================================
# FINAL SENSITIVITY CHECK:
# SHORTENING AFTER EXCLUDING 3 AMBIGUOUS PAIRS
# ============================================================

reviewed = pd.read_csv(
    OUT_DIR /
    "perturbation_semantic_audit_REVIEWED_FINAL.csv"
)

reviewed["valid_semantics"] = pd.to_numeric(
    reviewed["valid_semantics"],
    errors="coerce"
)

# Select only the shortening audit rows.
short_review = (
    reviewed[
        reviewed["condition"] == "shortening"
    ]
    .reset_index(drop=True)
)

print("Shortening audit rows:", len(short_review))
print(
    short_review["valid_semantics"]
    .value_counts(dropna=False)
)

valid_short_ids = short_review.index[
    short_review["valid_semantics"] == 1
].to_numpy()

uncertain_short_ids = short_review.index[
    short_review["valid_semantics"] == 2
].to_numpy()

print("\nValid shortening pairs:", len(valid_short_ids))
print("Excluded ambiguous pairs:", len(uncertain_short_ids))
print(
    "Excluded local row IDs:",
    uncertain_short_ids.tolist()
)

# Safety checks
assert len(short_review) == 116
assert len(valid_short_ids) == 113
assert len(uncertain_short_ids) == 3


def shortening_sensitivity(model_name, seed=42):

    clean_q = get_query_block(
        model_name,
        seed,
        "clean_short"
    ).reset_index(drop=True)

    pert_q = get_query_block(
        model_name,
        seed,
        "shortening"
    ).reset_index(drop=True)

    assert len(clean_q) == 116
    assert len(pert_q) == 116

    clean_q = (
        clean_q.iloc[valid_short_ids]
        .reset_index(drop=True)
    )

    pert_q = (
        pert_q.iloc[valid_short_ids]
        .reset_index(drop=True)
    )

    clean_correct = (
        clean_q["correct"]
        .astype(bool)
        .to_numpy()
    )

    pert_correct = (
        pert_q["correct"]
        .astype(bool)
        .to_numpy()
    )

    harmed = int(
        np.sum(
            clean_correct &
            ~pert_correct
        )
    )

    helped = int(
        np.sum(
            ~clean_correct &
            pert_correct
        )
    )

    discordant = harmed + helped

    if discordant > 0:
        p_value = binomtest(
            min(harmed, helped),
            n=discordant,
            p=0.5,
            alternative="two-sided"
        ).pvalue
    else:
        p_value = 1.0

    clean_acc = clean_correct.mean()
    pert_acc = pert_correct.mean()

    margin_auc_clean = (
        roc_auc_score(
            clean_correct.astype(int),
            clean_q["margin"]
        )
        if len(np.unique(clean_correct)) == 2
        else np.nan
    )

    margin_auc_pert = (
        roc_auc_score(
            pert_correct.astype(int),
            pert_q["margin"]
        )
        if len(np.unique(pert_correct)) == 2
        else np.nan
    )

    margin_aurc_clean = aurc_from_score(
        clean_correct,
        clean_q["margin"]
    )

    margin_aurc_pert = aurc_from_score(
        pert_correct,
        pert_q["margin"]
    )

    return {
        "Model": model_name,
        "Seed": seed,
        "N": len(clean_q),
        "Clean accuracy": clean_acc,
        "Shortened accuracy": pert_acc,
        "Accuracy change":
            pert_acc - clean_acc,
        "Accuracy drop":
            clean_acc - pert_acc,
        "Harmed": harmed,
        "Helped": helped,
        "McNemar p": p_value,
        "Clean margin AUROC":
            margin_auc_clean,
        "Short margin AUROC":
            margin_auc_pert,
        "Clean margin AURC":
            margin_aurc_clean,
        "Short margin AURC":
            margin_aurc_pert,
    }


shortening_sensitivity_results = pd.DataFrame([
    shortening_sensitivity("LR", 42),
    shortening_sensitivity("Calibrated SVM", 42),
    shortening_sensitivity("Hybrid", 42),
    shortening_sensitivity("DistilBERT", 42)
])

display(
    shortening_sensitivity_results.round(4)
)

shortening_sensitivity_results.to_csv(
    OUT_DIR /
    "shortening_semantic_sensitivity.csv",
    index=False
)

Shortening audit rows: 116
valid_semantics
1    113
2      3
Name: count, dtype: int64

Valid shortening pairs: 113
Excluded ambiguous pairs: 3
Excluded local row IDs: [5, 7, 78]


,Model,Seed,N,Clean accuracy,Shortened accuracy,Accuracy change,Accuracy drop,Harmed,Helped,McNemar p,Clean margin AUROC,Short margin AUROC,Clean margin AURC,Short margin AURC
0,LR,42,113,0.4336,0.4779,0.0442,-0.0442,7,12,0.3593,0.6881,0.6667,0.3955,0.3773
1,Calibrated SVM,42,113,0.5398,0.5398,0.0000,0.0000,14,14,1.0000,0.7238,0.6107,0.2617,0.3346
2,Hybrid,42,113,0.7965,0.7168,-0.0796,0.0796,11,2,0.0225,0.8483,0.8287,0.0559,0.1003
3,DistilBERT,42,113,0.7257,0.6991,-0.0265,0.0265,5,2,0.4531,0.8662,0.8511,0.0884,0.1066
